# Graph clustering of Amazon books: reader communities for long-tail recommendations

**02807 Computational Tools for Data Science - final project, graph-clustering part.**

Question: do reader communities found by graph clustering (on graphs of co-readers, co-liked books and,
when available, review-text similarity) recommend long-tail books better than purchase overlap alone?

This notebook runs the whole graph-clustering part end to end:

1. preprocessing (shared with the group): time split, iterative k-core, "liked" flag
2. sparse user x book matrix
3. book-book graphs via sparse products (co-reader, co-liked, external edge lists)
4. edge weighting - raw counts, Jaccard, PPMI (**implemented by us**)
5. sparsification - disparity-filter backbone and top-k baseline (**implemented by us**)
6. clustering - Louvain (library, resolution and seed sweep) and spectral clustering (**implemented by us**)
7. community evaluation - modularity (**ours**, checked against libraries), sizes, NMI vs genres, mixed-genre communities
8. community-based recommender vs "most popular" and "readers also bought" baselines, by popularity group

Reusable code lives in `src/ctds/graph/` (small functions with docstrings); the notebook only orchestrates
and explains. Design decisions and their reasons: `docs/graph_clustering_decisions.md`.

### How to run

1. Put the data next to the repo: `../data/books_50pct/books_50pct_users_reviews.parquet` and
   `books_50pct_users_meta.parquet` (unzip `books_50pct.zip`), or set `CTDS_DATA_DIR`.
2. `uv sync` in the repo, then open this notebook with the project's kernel.
3. Run with `SAMPLE = True` first (15% of users, a few minutes), then `SAMPLE = False` for the full
   data (~20-30 min on a 16 GB laptop). From a terminal:
   `CTDS_SAMPLE=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/graph_clustering.ipynb`
4. Tables are written to `docs/results/<run>/`, figures to `docs/figures/<run>/`. Intermediate data is
   cached as Parquet / npz in `../data/cache/graph/<run>/` (never in the repo).

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))  # also works when macOS hides the venv's .pth file

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import display

from ctds.graph.config import Config, sample_config
from ctds.graph.data import prepare
from ctds.graph.matrix import Bipartite, build_bipartite
from ctds.graph.metadata import load_book_metadata
from ctds.graph.build import cooccurrence_graph, graph_from_edge_list, user_weights
from ctds.graph.weights import apply_weighting
from ctds.graph.sparsify import (alpha_sweep, backbone_stats, disparity_alpha_for_edges, disparity_filter,
                                 giant_component, top_k_neighbours)
from ctds.graph.cluster import (louvain, modularity, networkx_modularity, resolution_sweep, spectral_clustering,
                                spectral_embedding, summarise_sweep, to_igraph)
from ctds.graph.evaluate import community_genre_table, community_sizes, genre_nmi, size_summary
from ctds.graph.recommend import (CommunityRecommender, HybridRecommender, ItemKNNRecommender, PopularityRecommender,
                                  book_popularity, evaluate, make_test_set, paired_bootstrap, popularity_groups,
                                  series_onehot, split_test_users)
from ctds.graph.utils import process_peak_rss_gb, runtime_table, timed
from ctds.graph import plotting as P

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.precision", 4)
P.set_style()

# ---- main switch -------------------------------------------------------------------------------
SAMPLE = os.environ.get("CTDS_SAMPLE", "1") == "1"   # True: quick sample run; False: full data
SEED = 42
cfg = sample_config(seed=SEED) if SAMPLE else Config(seed=SEED)

RUN = cfg.tag
RESULTS = REPO / "docs" / "results" / RUN; RESULTS.mkdir(parents=True, exist_ok=True)
FIGS = REPO / "docs" / "figures" / RUN; FIGS.mkdir(parents=True, exist_ok=True)
SUMMARY = {"run": RUN, "config": cfg.as_dict()}   # headline numbers, saved as JSON at the end

def save_table(df, name):
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    return df

print("run:", RUN); print("data:", cfg.data_dir); print("cache:", cfg.run_dir)

In [ ]:
# ---- parameters of the graph-clustering part ---------------------------------------------------
SOURCES = {"co-reader": "read", "co-liked": "liked"}   # which user x book matrix each graph comes from
WEIGHTINGS = ["raw", "jaccard", "ppmi"]
USER_WEIGHTING = "log"        # 1/log(1+d_u) down-weighting of heavy users (see build.user_weights)
MIN_SHARED = 2                # keep an edge only if >= 2 users are shared
TOP_K = 10                    # top-k baseline; the disparity backbone is matched to its edge budget
ALPHAS = [0.3, 0.1, 0.05, 0.01]
RESOLUTIONS = [0.5, 1, 2, 5, 10, 20]
SEEDS = [0, 1, 2, 3, 4]
MAIN = ("co-liked", "jaccard", "disparity")   # variant used for the detailed clustering analysis
SPECTRAL_KS = [20, 40, 60]
TEXT_GRAPH_PATH = None        # teammate's LSH/SBERT edge list (Parquet: book_a, book_b, weight), when it exists
K = 10                        # top-10 recommendations

## 1. Preprocessing (shared with the group)

The data is a user-based pull of ~50% of Amazon Books reviewers with **all** their book reviews (Amazon 2014
release, reviews 1996 - July 2014; see the decisions document). Agreed preprocessing:

- all years; **cut-off 2014-01-01** - graphs are built from earlier reviews, Jan-Jul 2014 is the test period
- **iterative k-core**: users >= 5 reviews and books >= 15 reviews, repeated until both hold (sample run: books >= 10)
- **liked** = rating >= 4 (and, once the sentiment column exists, sentiment not negative)
- optional duplicate-edition mapping from the LSH part (identity until it exists)

In [ ]:
with timed("1 preprocessing"):
    data = prepare(cfg)
train, test, kcore_log = data["train"], data["test"], data["kcore_log"]
display(save_table(kcore_log, "kcore_log"))

split = pd.DataFrame({
    "part": ["train (< cut-off)", "test (>= cut-off)"],
    "reviews": [len(train), len(test)],
    "users": [train.user_id.nunique(), test.user_id.nunique()],
    "books": [train.item_id.nunique(), test.item_id.nunique()],
    "liked_pct": [100 * train.liked.mean(), 100 * test.liked.mean()],
})
display(save_table(split, "train_test_split"))
SUMMARY["kcore"] = kcore_log.iloc[-1][["reviews", "users", "books"]].astype(int).to_dict()
SUMMARY["kcore_iterations"] = int(kcore_log.iteration.max())

In [ ]:
both = pd.concat([train, test])
years = pd.to_datetime(both.timestamp, unit="s").dt.year.value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
axes[0].bar(years.index, years.values, color=P.PALETTE[0], width=0.8)
axes[0].axvline(2013.5, color=P.TEXT_2, lw=1, ls="--"); axes[0].text(2013.3, years.max() * 0.95, "cut-off", ha="right", color=P.TEXT_2, fontsize=8)
axes[0].set_title("Reviews per year (after k-core)"); axes[0].set_ylabel("reviews")
for ax, col, name in [(axes[1], "user_id", "user"), (axes[2], "item_id", "book")]:
    counts = train[col].value_counts().to_numpy()
    x = np.sort(counts); ccdf = 1 - np.arange(len(x)) / len(x)
    ax.loglog(x, ccdf, color=P.PALETTE[0])
    ax.set_title(f"Training reviews per {name} (CCDF)"); ax.set_xlabel(f"reviews per {name}"); ax.set_ylabel("share with >= x")
P.save(fig, FIGS, "01_data_distributions"); plt.show()

## 2. Sparse user x book matrix

Two CSR matrices from the training reviews: `read` (1 if the user reviewed the book) and `liked`
(1 if the review counts as liked). Integer index mappings are saved next to them so other parts can reuse them.

In [ ]:
with timed("2 bipartite matrix"):
    bp = build_bipartite(train)
    bp.save(cfg.run_dir)
meta = load_book_metadata(cfg, bp.book_ids)            # row i = book i
info = {"users": bp.n_users, "books": bp.n_books, "read_nnz": bp.read.nnz, "liked_nnz": bp.liked.nnz,
        "density_pct": 100 * bp.read.nnz / (bp.n_users * bp.n_books),
        "books_with_genre": int(meta.primary_genre.notna().sum()), "books_with_series": int(meta.series.notna().sum())}
display(pd.DataFrame([info]))
SUMMARY["bipartite"] = info

## 3. Book-book graphs via sparse products

The co-occurrence graph is $C = B^\top W B$ for the user x book matrix $B$ (`read` -> **co-reader** graph, `liked` ->
**co-liked** graph: only readers who liked *both* books count). It is computed in blocks of books and
thresholded per block, so no dense all-pairs matrix is ever formed.

**Heavy users.** A user with $d$ books adds $d(d-1)/2$ pairs, so a few heavy reviewers would dominate the edge
weights. Each user is weighted by $w_u = 1/\log(1+d_u)$ (like inverse document frequency / Adamic-Adar): heavy users
stay in the graph but count less. After the k-core the heaviest training user has ~2k books, so the product fits
in memory without a cap (`max_user_books` exists for other settings). We also require >= 2 shared users per edge.

In [ ]:
deg = np.diff(bp.read.indptr)
pairs = deg * (deg - 1) / 2
heavy = deg > 500
print(f"max books per user {deg.max()}, users > 500 books: {heavy.sum()} ({100 * heavy.mean():.2f}%), "
      f"share of co-occurrence pairs they create: {100 * pairs[heavy].sum() / pairs.sum():.1f}%")
w = user_weights(deg)
print(f"weight of a median user ({int(np.median(deg))} books): {1 / np.log1p(np.median(deg)):.3f}; of the heaviest: {w.min():.3f}")
SUMMARY["heavy_users"] = {"max_degree": int(deg.max()), "users_gt500": int(heavy.sum()),
                          "pair_share_gt500_pct": round(float(100 * pairs[heavy].sum() / pairs.sum()), 1)}

cographs = {}
for name, matrix in SOURCES.items():
    with timed(f"3 graph {name}"):
        cographs[name] = cooccurrence_graph(getattr(bp, matrix), user_weighting=USER_WEIGHTING,
                                            min_shared=MIN_SHARED, seed=SEED, name=name)
graph_table = pd.DataFrame([{"graph": n, "nodes": g.weights.shape[0], "edges": g.n_edges,
                             "isolated_books": int((np.diff(g.weights.indptr) == 0).sum()),
                             "mean_degree": g.weights.nnz / g.weights.shape[0]} for n, g in cographs.items()])
display(save_table(graph_table, "cooccurrence_graphs"))

**External graphs (review text).** The LSH / SBERT teammate's review-text similarity graph enters through
`graph_from_edge_list`: any `(book_a, book_b, weight)` table is mapped to our book index (optionally through the
canonical-edition map) and then goes through exactly the same sparsification, clustering and evaluation.
Until that file exists we check the adapter by round-tripping one of our own graphs through an edge list.

In [ ]:
U = sp.triu(cographs["co-liked"].weights, k=1).tocoo()
edge_list = pd.DataFrame({"book_a": bp.book_ids[U.row], "book_b": bp.book_ids[U.col], "weight": U.data})
roundtrip, report = graph_from_edge_list(edge_list, bp.book_ids)
print("adapter report:", report, "| identical to the original:", abs(roundtrip - cographs["co-liked"].weights).max() < 1e-6)

external = {}
if TEXT_GRAPH_PATH and Path(TEXT_GRAPH_PATH).exists():
    G_text, report = graph_from_edge_list(pd.read_parquet(TEXT_GRAPH_PATH), bp.book_ids)
    print("text graph:", report)
    external["text"] = G_text

## 4. Edge weighting (implemented by us)

- **raw**: (user-weighted) number of shared readers - dominated by popular books
- **Jaccard**: $|U_i \cap U_j| / |U_i \cup U_j|$ - corrects for both books' popularity
- **PPMI**: $\max(0, \log \frac{C_{ij}\,T}{r_i r_j})$ - how much more often the pair co-occurs than expected by chance;
  penalises hubs most strongly

The example below shows the nearest neighbours of the most-read book under each weighting. Two things to note:
the same book can appear twice (separate editions - what the LSH teammate's canonical mapping fixes), and PPMI
gives popular books rare neighbours (a book with ~15 readers that shares 2 of them gets a huge PMI). This is the
known rare-pair bias of PMI; context-distribution smoothing (0.75) did not remove it here, so we keep plain PPMI
and report it as a finding. For long-tail books PPMI and Jaccard agree closely.

In [ ]:
weighted = {}
for src, g in cographs.items():
    for scheme in WEIGHTINGS:
        with timed(f"4 weight {src} {scheme}"):
            weighted[(src, scheme)] = apply_weighting(g.weights, scheme, g.strength)
for name, G in external.items():
    weighted[(name, "raw")] = G

pop = book_popularity(bp)
title = lambda i: (meta.title.iloc[i] if pd.notna(meta.title.iloc[i]) else meta.item_id.iloc[i])[:55]
probe = int(np.argmax(pop))
rows = []
for scheme in WEIGHTINGS:
    row = weighted[("co-liked", scheme)][probe].tocoo()
    for rank, j in enumerate(row.col[np.argsort(-row.data)][:5], 1):
        rows.append({"weighting": scheme, "rank": rank, "neighbour": title(j), "readers": int(pop[j])})
print(f"Most-read book: {title(probe)} ({int(pop[probe])} readers)")
display(save_table(pd.DataFrame(rows).pivot(index="rank", columns="weighting", values="neighbour"), "example_neighbours"))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
for scheme in WEIGHTINGS:
    W = weighted[("co-liked", scheme)]
    # Correlation between an edge's weight and the popularity of its endpoints: lower = less popularity bias.
    U = sp.triu(W, k=1).tocoo()
    r = np.corrcoef(np.log(U.data + 1e-12), np.log(pop[U.row] * pop[U.col]))[0, 1]
    vals = np.log10(U.data[U.data > 0])
    ax.hist((vals - vals.min()) / (vals.max() - vals.min()), bins=60, histtype="step", lw=1.6,
            color=P.WEIGHTING_COLOURS[scheme], label=f"{scheme} (corr. with popularity {r:.2f})")
ax.set_title("Edge-weight distributions, co-liked graph"); ax.set_xlabel("log10 weight, rescaled to [0, 1]"); ax.set_ylabel("edges")
ax.legend(); P.save(fig, FIGS, "04_weight_distributions"); plt.show()

## 5. Sparsification (implemented by us)

**Disparity filter** (Serrano, Boguna & Vespignani 2009): for node $i$ with degree $k_i$ and strength $s_i$, an edge's
p-value is $\alpha_{ij} = (1 - w_{ij}/s_i)^{k_i - 1}$ - the chance that a uniformly random split of $i$'s strength
gives an edge at least this heavy. Edges significant for at least one endpoint are kept. This keeps the locally
important links of weak (long-tail) books instead of applying one global threshold.

**Top-k** baseline: keep each book's $k$ heaviest edges.

A fixed $\alpha$ means very different things for different weightings (PPMI weights are almost uniform around each
node, so almost nothing is "significant"), so for the comparison we choose $\alpha$ per graph such that the
backbone has the **same number of edges as top-10**.

In [ ]:
sweeps = []
for key, W in weighted.items():
    s = alpha_sweep(W, ALPHAS, name=f"{key[0]} {key[1]}")
    sweeps.append(s)
alpha_table = save_table(pd.concat(sweeps), "disparity_alpha_sweep")
display(alpha_table[["graph", "alpha", "edges", "edges_kept_pct", "nodes_kept_pct", "weight_kept_pct", "giant_component_pct"]])

fig, ax = plt.subplots(figsize=(6.5, 3.2))
for scheme in WEIGHTINGS:
    s = alpha_table[alpha_table.graph == f"co-liked {scheme}"]
    ax.plot(s.alpha, s.giant_component_pct, marker="o", color=P.WEIGHTING_COLOURS[scheme], label=scheme)
ax.set_xscale("log"); ax.set_xlabel("disparity-filter significance level alpha"); ax.set_ylabel("giant component (% of books)")
ax.set_title("Disparity backbone at fixed alpha, co-liked graph"); ax.legend()
P.save(fig, FIGS, "05_alpha_sweep"); plt.show()

In [ ]:
variants, backbone_rows = {}, []
for key, W in weighted.items():
    with timed(f"5 sparsify {key[0]} {key[1]}"):
        K_ = top_k_neighbours(W, TOP_K)
        alpha = disparity_alpha_for_edges(W, K_.nnz // 2)
        D_ = disparity_filter(W, alpha)
    for method, G in [("disparity", D_), (f"top{TOP_K}", K_)]:
        variants[(*key, method)] = G
        backbone_rows.append({"source": key[0], "weighting": key[1], "sparsifier": method,
                              "alpha": alpha if method == "disparity" else np.nan,
                              **backbone_stats(W, G)})
    del K_, D_
backbones = save_table(pd.DataFrame(backbone_rows).drop(columns="graph"), "backbones")
display(backbones)
del weighted   # the dense-ish weighted graphs are no longer needed

## 6. Clustering

### 6.1 Louvain: resolution sweep and stability across seeds (main variant)

Louvain (igraph's C implementation of `community_multilevel`) maximises modularity with resolution $\gamma$:
larger $\gamma$ gives more, smaller communities. Each resolution is run with 5 seeds; stability is the mean pairwise
NMI / ARI between the seeds' partitions. Modularity is computed with **our** function.

In [ ]:
D = variants[MAIN]
with timed("6 louvain sweep"):
    sweep, partitions = resolution_sweep(D, RESOLUTIONS, SEEDS)
sweep_summary = save_table(summarise_sweep(sweep), "louvain_resolution_sweep")
display(sweep_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(sweep_summary.resolution, sweep_summary.modularity, marker="o", label="modularity (gamma = 1)")
axes[0].plot(sweep_summary.resolution, sweep_summary.seed_stability_nmi, marker="s", label="stability across seeds (NMI)")
axes[0].set_xscale("log"); axes[0].set_xlabel("resolution gamma"); axes[0].set_ylim(0, 1); axes[0].legend()
axes[0].set_title("Quality and stability")
axes[1].plot(sweep_summary.resolution, sweep_summary.n_communities_ge10, marker="o", color=P.PALETTE[0])
axes[1].set_xscale("log"); axes[1].set_xlabel("resolution gamma"); axes[1].set_ylabel("communities with >= 10 books")
axes[1].set_title("Number of communities")
P.save(fig, FIGS, "06_resolution_sweep"); plt.show()

### 6.2 Sanity checks: our modularity vs libraries; igraph vs networkx Louvain

In [ ]:
lab_main = partitions[(1, 0)]
checks = {"modularity_ours": modularity(D, lab_main),
          "modularity_igraph": to_igraph(D).modularity(lab_main.tolist(), weights="weight"),
          "modularity_networkx": networkx_modularity(D, lab_main)}
with timed("6 networkx louvain"):
    lab_nx = louvain(D, 1.0, seed=0, backend="networkx")
from sklearn.metrics import normalized_mutual_info_score as nmi
has_edge = np.diff(D.indptr) > 0
checks.update({"networkx_louvain_modularity": modularity(D, lab_nx),
               "nmi_igraph_vs_networkx_partition": nmi(lab_main[has_edge], lab_nx[has_edge])})
display(pd.DataFrame([checks]).T.rename(columns={0: "value"}))
SUMMARY["modularity_checks"] = {k: float(v) for k, v in checks.items()}

### 6.3 Spectral clustering (implemented by us)

Normalised spectral clustering (Ng, Jordan & Weiss): the $k$ eigenvectors of the normalised Laplacian
$L_{sym} = I - D^{-1/2} W D^{-1/2}$ with the smallest eigenvalues (scipy `eigsh` on $D^{-1/2} W D^{-1/2}$),
row-normalised, then k-means. It runs on the **giant component** of the main backbone (the Laplacian of a
disconnected graph has one zero eigenvalue per component). This is feasible at full size because the backbone is sparse.

In [ ]:
gc = giant_component(D)
Wg = D[gc][:, gc]
with timed("6 spectral embedding"):
    eigvals, emb = spectral_embedding(Wg, max(SPECTRAL_KS), seed=SEED)
meta_gc = meta.iloc[gc].reset_index(drop=True)
lab_gc = lab_main[gc]
rows = [{"method": "Louvain (gamma=1)", "k": len(np.unique(lab_gc)), "modularity": modularity(Wg, lab_gc),
         **{k: v for k, v in genre_nmi(lab_gc, meta_gc).items() if k in ("nmi", "nmi_random_baseline")}}]
for k in SPECTRAL_KS:
    with timed(f"6 spectral k={k}"):
        lab_sp = spectral_clustering(Wg, k, seed=SEED, embedding=emb)
    rows.append({"method": "spectral (ours)", "k": k, "modularity": modularity(Wg, lab_sp),
                 **{kk: v for kk, v in genre_nmi(lab_sp, meta_gc).items() if kk in ("nmi", "nmi_random_baseline")}})
spectral_table = save_table(pd.DataFrame(rows), "spectral_vs_louvain")
print(f"giant component: {len(gc)} books, {Wg.nnz // 2} edges")
display(spectral_table)

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(np.arange(1, len(eigvals) + 1), eigvals, marker="o", ms=3, color=P.PALETTE[0])
ax.set_xlabel("index"); ax.set_ylabel("eigenvalue of L_sym"); ax.set_title("Smallest Laplacian eigenvalues, giant component")
P.save(fig, FIGS, "06_spectral_eigenvalues"); plt.show()
SUMMARY["giant_component_books"] = int(len(gc))

### 6.4 Louvain on every graph variant

All graphs at $\gamma = 1$, seed 0. NMI is computed on the books that have a genre label (see decisions D6), with a
random-permutation baseline.

In [ ]:
labels, rows = {}, []
for key, G in variants.items():
    with timed(f"6 louvain {' '.join(key)}"):
        lab = louvain(G, 1.0, seed=0)
    labels[key] = lab
    nodes = np.flatnonzero(np.diff(G.indptr) > 0)
    rows.append({"source": key[0], "weighting": key[1], "sparsifier": key[2], "modularity": modularity(G, lab),
                 **size_summary(lab, nodes), **genre_nmi(lab, meta, nodes)})
variant_table = save_table(pd.DataFrame(rows), "louvain_variants")
display(variant_table[["source", "weighting", "sparsifier", "modularity", "n_ge10", "largest_pct", "pct_nodes_in_ge10",
                       "nmi", "nmi_single_genre", "nmi_random_baseline", "ari"]])

## 7. Community evaluation (main variant)

Size distribution, and the genre make-up of each community. A community is **mixed** when its second most common
genre still covers >= 25% of its genre-labelled books - reader communities that cut across Amazon's genre tree.

In [ ]:
nodes_main = np.flatnonzero(np.diff(D.indptr) > 0)
sizes = community_sizes(lab_main, nodes_main).to_numpy()
fig, ax = plt.subplots(figsize=(6.5, 3.2))
x = np.sort(sizes); ax.loglog(x, 1 - np.arange(len(x)) / len(x), marker="o", ms=3, lw=1, color=P.PALETTE[0])
ax.set_xlabel("community size (books)"); ax.set_ylabel("share of communities with size >= x")
ax.set_title(f"Community sizes, {' / '.join(MAIN)}, gamma = 1")
P.save(fig, FIGS, "07_community_sizes"); plt.show()

genre_table = save_table(community_genre_table(lab_main, meta, D, min_labelled=15), "community_genres")
print(f"{len(genre_table)} communities with >= 15 labelled books; {int(genre_table.mixed.sum())} are mixed")
display(genre_table.head(15))
display(genre_table[genre_table.mixed].sort_values("entropy", ascending=False).head(12))
SUMMARY["main_partition"] = {**size_summary(lab_main, nodes_main), **genre_nmi(lab_main, meta, nodes_main),
                             "modularity": float(modularity(D, lab_main)),
                             "mixed_communities": int(genre_table.mixed.sum()), "communities_ge15_labelled": len(genre_table)}

## 8. Community-based recommender

For each test user we recommend 10 books they had not read before the cut-off and evaluate against the books they
**liked in Jan-Jul 2014** (only books that exist in the training graph; brand-new books are the cold-start part).
The same test users and items are used for every variant.

Recommenders:
- **most popular** - baseline
- **item-kNN (co-reader)** - "readers also bought": sum of raw co-reader counts with the user's liked books. This is
  *purchase overlap alone*, the baseline the project question is about
- **community** - books from the communities of the user's liked books. The within-community ranking is a design
  choice: weighted degree inside the community (`strength`), popularity, or `personal` (backbone links to the
  user's own liked books, restricted to the same community)
- **hybrid** - $(1-\lambda)\,$item-kNN $+ \lambda\,$community (both row-normalised): do communities add signal to overlap?

**Popularity groups:** books sorted by training readers and cut into head / mid / tail so that each holds one third
of all training interactions. Per group, only the user's relevant books in that group count.

**No tuning on the test set:** 20% of the test users are a validation set used to choose the resolution, the ranking
and $\lambda$ on the main variant; all reported numbers are on the remaining 80%.

In [ ]:
with timed("8 test set"):
    test_all = make_test_set(test, bp)
    val, final = split_test_users(test_all, val_frac=0.2, seed=SEED)
groups = popularity_groups(pop)
S = series_onehot(meta["series"])
group_table = pd.DataFrame({
    "group": ["head", "mid", "tail"],
    "books": [int((groups == g).sum()) for g in ("head", "mid", "tail")],
    "min_readers": [int(pop[groups == g].min()) for g in ("head", "mid", "tail")],
    "test_items": [int(test_all.relevant[:, np.flatnonzero(groups == g)].nnz) for g in ("head", "mid", "tail")],
})
display(save_table(group_table, "popularity_groups"))
print(f"test users: {len(test_all.users)} (validation {len(val.users)}, final {len(final.users)}); "
      f"relevant items: {test_all.relevant.nnz}")
SUMMARY["test"] = {"users": int(len(test_all.users)), "val_users": int(len(val.users)), "final_users": int(len(final.users)),
                   "relevant": int(test_all.relevant.nnz), "groups": group_table.to_dict("records")}

knn = ItemKNNRecommender(cographs["co-reader"].shared, pop, name="item-kNN (co-reader)")
popular = PopularityRecommender(pop)

In [ ]:
def headline(rec, ts, series=None):
    r = evaluate(rec, bp, ts, groups, series, k=K)
    return r

val_rows = []
def val_eval(rec, **info):
    r = headline(rec, val).set_index("group")
    val_rows.append({**info, "recall@10": r.loc["all", "recall@10"], "ndcg@10": r.loc["all", "ndcg@10"],
                     "recall_mid": r.loc["mid", "recall@10"], "recall_tail": r.loc["tail", "recall@10"],
                     "coverage": r.loc["all", "coverage"]})

with timed("8 validation grid"):
    val_eval(knn, model="item-kNN")
    for res in [1, 5, 20]:
        lab = partitions[(res, 0)] if (res, 0) in partitions else louvain(D, res, seed=0)
        for within in ["strength", "popularity", "personal"]:
            val_eval(CommunityRecommender(lab, D, pop, within=within), model="community", resolution=res, within=within)
val_grid = save_table(pd.DataFrame(val_rows), "validation_grid")
display(val_grid)

best = val_grid[val_grid.model == "community"].sort_values("recall@10", ascending=False).iloc[0]
BEST_RES, BEST_WITHIN = float(best.resolution), best.within
print(f"chosen on validation: resolution {BEST_RES}, within-community ranking '{BEST_WITHIN}'")

In [ ]:
lab_best = partitions.get((BEST_RES, 0), None)
if lab_best is None:
    lab_best = louvain(D, BEST_RES, seed=0)
comm_best = CommunityRecommender(lab_best, D, pop, within=BEST_WITHIN)
lam_rows = []
with timed("8 validation lambda"):
    for lam in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
        r = headline(HybridRecommender(knn, comm_best, lam), val).set_index("group")
        lam_rows.append({"lambda": lam, **{f"recall_{g}": r.loc[g, "recall@10"] for g in ["all", "head", "mid", "tail"]},
                         "ndcg@10": r.loc["all", "ndcg@10"]})
lam_table = save_table(pd.DataFrame(lam_rows), "validation_lambda")
display(lam_table)
BEST_LAM = float(lam_table.loc[lam_table.recall_all.idxmax(), "lambda"])
print("chosen lambda:", BEST_LAM)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
for g, col in [("all", P.TEXT), ("mid", P.GROUP_COLOURS["mid"]), ("tail", P.GROUP_COLOURS["tail"])]:
    ax.plot(lam_table["lambda"], lam_table[f"recall_{g}"], marker="o", color=col, label=g)
ax.set_xlabel("lambda (0 = item-kNN only, 1 = community only)"); ax.set_ylabel("recall@10 (validation)")
ax.set_title("Adding the community signal to purchase overlap"); ax.legend()
P.save(fig, FIGS, "08_hybrid_lambda"); plt.show()
SUMMARY["chosen"] = {"resolution": BEST_RES, "within": BEST_WITHIN, "lambda": BEST_LAM}

### 8.1 Final evaluation on the held-out users

Every graph variant is clustered with the chosen resolution and evaluated with the chosen ranking, as a pure
community recommender and as a hybrid with item-kNN. "No same-series" removes books from series the user already
read (series continuations are easy hits that inflate scores; this replaces the same-author check because the
metadata has no author).

In [ ]:
results, per_user = [], {}
def final_eval(rec, **info):
    r, pu = evaluate(rec, bp, final, groups, S, k=K, return_per_user=True)
    per_user[rec.name] = pu
    for k_, v in info.items():
        r[k_] = v
    results.append(r)

with timed("8 final baselines"):
    final_eval(popular, family="baseline", source="-", weighting="-", sparsifier="-")
    final_eval(knn, family="baseline", source="co-reader", weighting="raw", sparsifier="none")
for key, G in variants.items():
    with timed(f"8 final {' '.join(key)}"):
        lab = louvain(G, BEST_RES, seed=0) if key != MAIN else lab_best
        comm = CommunityRecommender(lab, G, pop, within=BEST_WITHIN, name=f"community {' / '.join(key)}")
        info = dict(source=key[0], weighting=key[1], sparsifier=key[2])
        final_eval(comm, family="community", **info)
        final_eval(HybridRecommender(knn, comm, BEST_LAM, name=f"hybrid {' / '.join(key)}"), family="hybrid", **info)
        final_eval(ItemKNNRecommender(G, pop, name=f"backbone-kNN {' / '.join(key)}"), family="backbone-kNN", **info)
final_results = save_table(pd.concat(results, ignore_index=True), "recommender_results")

In [ ]:
def wide(df, setting, metric="recall@10"):
    t = df[df.setting == setting].pivot_table(index=["family", "recommender"], columns="group", values=metric)
    cov = df[(df.setting == setting) & (df.group == "all")].set_index(["family", "recommender"])["coverage"]
    return t[["all", "head", "mid", "tail"]].assign(coverage=cov).sort_values("all", ascending=False)

for setting in ["all books", "no same-series"]:
    for metric in ["recall@10", "ndcg@10", "hit@10"]:
        t = wide(final_results, setting, metric)
        save_table(t.reset_index(), f"final_{metric.replace('@', '')}_{setting.replace(' ', '_')}")
        if metric == "recall@10":
            print(f"=== {metric}, {setting} ===")
            display(t.round(4))

main_name = " / ".join(MAIN)
key_models = ["most popular", "item-kNN (co-reader)", f"community {main_name}", f"hybrid {main_name}", f"backbone-kNN {main_name}"]
t = final_results[(final_results.setting == "no same-series") & final_results.recommender.isin(key_models) & (final_results.group != "all")]
fig, ax = plt.subplots(figsize=(8, 3.4))
width = 0.16
for i, m in enumerate(key_models):
    vals = t[t.recommender == m].set_index("group").reindex(["head", "mid", "tail"])["recall@10"]
    ax.bar(np.arange(3) + (i - 2) * width, vals.values, width=width - 0.02, color=P.PALETTE[i], label=m)
ax.set_xticks(range(3), ["head", "mid", "tail"]); ax.set_ylabel("recall@10 (no same-series)")
ax.set_title("Recall by book popularity group, held-out users"); ax.legend(fontsize=7, ncol=2)
P.save(fig, FIGS, "08_recall_by_group"); plt.show()

hl = final_results[(final_results.group == "all")].set_index(["setting", "recommender"])
SUMMARY["final_headline"] = {f"{s} | {m}": {k_: float(hl.loc[(s, m), k_]) for k_ in ["recall@10", "ndcg@10", "hit@10", "coverage"]}
                             for s in ["all books", "no same-series"] for m in key_models if (s, m) in hl.index}

### 8.2 Are the differences significant?

Paired bootstrap over the held-out users (2,000 resamples, 95% CI) of the per-user recall@10, without same-series
books. The key comparison for the role of clustering is **community vs backbone-kNN on the same graph**: both use the
same weighted backbone, the only difference is that the community recommender restricts candidates to the user's
communities.

In [ ]:
comparisons = [(f"community {main_name}", "item-kNN (co-reader)"),
               (f"hybrid {main_name}", "item-kNN (co-reader)"),
               (f"community {main_name}", f"backbone-kNN {main_name}"),
               (f"backbone-kNN {main_name}", "item-kNN (co-reader)"),
               ("item-kNN (co-reader)", "most popular")]
rows = []
for a, b in comparisons:
    for g in ["all", "head", "mid", "tail"]:
        key = ("no same-series", g)
        rows.append({"a": a, "b": b, "group": g,
                     **paired_bootstrap(per_user[a][key]["recall"], per_user[b][key]["recall"], seed=SEED)})
significance = save_table(pd.DataFrame(rows), "significance_recall")
display(significance.round(4))
SUMMARY["significance"] = significance.to_dict("records")

## 9. Runtime and memory

In [ ]:
rt = save_table(runtime_table(), "runtimes")
display(rt)
print(f"process peak RSS: {process_peak_rss_gb():.2f} GB")
SUMMARY["peak_rss_gb"] = round(process_peak_rss_gb(), 2)
SUMMARY["total_runtime_min"] = round(rt.seconds.sum() / 60, 1)
(RESULTS / "summary.json").write_text(json.dumps(SUMMARY, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("saved", RESULTS / "summary.json")

## 10. Summary

The numbers in this notebook are collected in `docs/graph_clustering_results.md` (written from the full-data run).
Course methods: Louvain, spectral clustering, modularity. Outside the course: PPMI, Jaccard weighting of a
projected bipartite graph, the disparity-filter backbone.